# corpus

> Build the labeled corpus: weak positives from scored replies, mixed human negatives

In [ ]:
#| default_exp corpus

In [ ]:
#| hide
from nbdev.showdoc import *

Training the learned scorer needs labeled text, and no human has labeled any. This module builds the corpus by weak supervision: the rules pick confident slop from my own reply history, human-authored sources supply the clean side, and the ambiguous middle joins neither. The one human-labeled set, replies that drew a complaint in a session, is written separately and reserved for evaluation. Rows carry their source, so any experiment can slice, reweight, or hold out a source later.

In [ ]:
#| export
import base64, json, httpx
from collections import Counter
from datetime import datetime, timedelta
from fastcore.utils import *
from fastcore.xdg import xdg_state_home
from slopometer.core import *
from slopometer.segment import *
from slopometer.lexicon import *
from slopometer.syntax import *
from slopometer.para import *
from slopometer.score import *
from slopometer.features import *

In [ ]:
from fastcore.test import *

## The label-side scorer

The rules select the positives, and the forest's features must stay independent of that selection. Most features are safe by construction: burstiness, sentence lengths, part-of-speech rates, specifics density, and the referential measures share no vocabulary with any rule. The connective rates do: `furthermore` sits in both the connective sets and the filler-transitions lexicon. A word in both places would let the label leak into a feature. `label_overlap` computes the shared words mechanically, and `label_density` scores with findings on those words dropped. The labels come from `label_density`; users keep `score_text` unchanged.

In [ ]:
#| export
from slopometer.features import _conn

def label_overlap():
    "Words shared between the connective feature sets and any rule lexicon"
    lexs = set(banned) | set(hedges) | set(noting) | set(transitions) | set(wordy)
    return set().union(*_conn.values()) & lexs

def label_density(txt):
    "Density with findings on `label_overlap` words dropped: the label-side scorer"
    ov = label_overlap()
    fs = [f for f in run_rules(txt) if f.text.lower() not in ov]
    return round(100*sum(f.weight for f in fs)/max(prose_words(segment(txt)), 1), 1)

In [ ]:
ov = label_overlap()
s = 'Furthermore, the cache is seamless in some cases.'
assert 'furthermore' in ov
test_eq(label_density(s), score_text(s).density - 300/len(s.split()))
label_density(s), score_text(s).density

## Replies from the session mirror

`llmsurgery` keeps an ipynb mirror of every session on the machine. Each prompt message carries its reply, and a reply mixes mid-turn notes, tool blocks, and the final message. The user only reacts to the final message, and the labels must attach to what the user read. `reply_final` therefore keeps the text after the last tool block and drops the rest. `mirror_replies` walks recent mirrors and yields one row per qualifying final.

In [ ]:
#| export
_toolblock = re.compile(r'```json \{\.tool\}.*?```', re.S)

def reply_final(out):
    "The text after the last tool block: the final message the user read"
    return _toolblock.split(out)[-1].strip()

def _mirror_prompts(days, path_re):
    "Prompt messages from recent mirrored sessions, each with its mirror path"
    from llmsurgery.mirror import index
    from aidialog.dlgskill import find_msgs
    from rgapi.skill import fd
    root = index().root
    cutoff = datetime.now() - timedelta(days=days)
    for p in fd(root, ext='ipynb', path_re=path_re):
        if p.mtime > cutoff:
            yield from ((p, m) for m in find_msgs(msg_type='prompt', context=0, dlg=str(root/p)))

def mirror_replies(
    days=21, # How far back to walk the mirror
    min_words=50, # Word floor per final
    path_re='-aai-ws', # Mirror filename filter: this machine's workspace projects
):
    "One (text, source, ref) row per qualifying reply final in recent mirrored sessions"
    rows = []
    for p, m in _mirror_prompts(days, path_re):
        txt = reply_final(m.out or '')
        if len(txt.split()) >= min_words: rows.append(dict(text=txt, source='reply', ref=f'{p}#{m.id}'))
    return rows


In [ ]:
r = 'Plan first.\n\n```json {.tool}\n{"name": "execute"}\n```\n\nA mid-turn note.\n\n```json {.tool}\n{"name": "execute"}\n```\n\nThe final message the user actually read.'
test_eq(reply_final(r), 'The final message the user actually read.')
reply_final(r)

Run on this machine, `mirror_replies()` returns 5,400 rows from the last three weeks of workspace sessions. The cell stays unevaluated on the docs build: the mirror is private and machine-local.

In [ ]:
#| eval: false
reps = mirror_replies()
len(reps)

## Paragraphs from documents

Every document source reduces the same way: segment the markdown, keep prose paragraphs over the word floor. `doc_paras` is that reduction, and each later fetcher wraps it around one source. The ref carries the file offset, so a row leads back to its paragraph.

In [ ]:
#| export
def doc_paras(txt, source, ref, min_words=50):
    "One row per prose paragraph of `min_words`+ words in markdown `txt`"
    return [dict(text=b.txt, source=source, ref=f'{ref}@{b.start}')
        for b in segment(txt) if b.kind=='prose' and len(b.txt.split()) >= min_words]

def readme_paras(root='~/aai-ws', source='readme', min_words=50):
    "Rows from every project README under `root`"
    from rgapi.skill import fd
    return [r for p in fd(root, glob='README.md', max_depth=2)
        for r in doc_paras((Path(root).expanduser()/p).read_text(), source, str(p), min_words)]

In [ ]:
ps = doc_paras(Path('../README.md').read_text(), 'readme', 'README.md')
assert all(len(r['text'].split()) >= 50 for r in ps)
len(ps), ps[0]['ref']

## The user's own requests

Long typed prompts are in-medium negatives: chat text, same topics, human-authored. Two filters guard the label. Prompts carrying harness markers (hook feedback, command output) were not typed and are skipped. Prompts pasted from AI output are indistinguishable by marker, and the low-density cut in `build_corpus` is what excludes them.

In [ ]:
#| export
_notuser = ('Stop hook feedback', '<bash-', '<command-', 'system-reminder', '<local-command')

def user_requests(
    days=90, # How far back to walk the mirror
    min_words=50, # Word floor per prompt
    path_re='-aai-ws', # Mirror filename filter
):
    "One row per long typed user prompt in recent mirrored sessions"
    return [dict(text=m.content, source='request', ref=f'{p}#{m.id}')
        for p, m in _mirror_prompts(days, path_re)
        if len((m.content or '').split()) >= min_words and not any(t in m.content for t in _notuser)]

In [ ]:
#| eval: false
ur = user_requests()
len(ur)

## READMEs from other people, before ChatGPT

Style diversity on the negative side needs authors other than one person. GitHub search returns the most-starred repos created before the cutoff, and for each the README as it stood at the cutoff date. Text written before November 2022 cannot have been written by a chat model, and mid-2022 leaves comfortable margin. Repo selection is mechanical (stars), not curated, and the quality filter is the same density cut every negative passes.

In [ ]:
#| export
async def gh_readme_asof(api, owner, repo, date='2022-06-01'):
    "The text of `owner/repo`'s README as of `date`, or None when none exists"
    cs = await api.repos.list_commits(owner=owner, repo=repo, path='README.md', until=f'{date}T00:00:00Z', per_page=1)
    if not cs: return None
    from fastspec.errors import APIError
    try: r = await api.repos.get_content(owner=owner, repo=repo, path='README.md', ref=cs[0].sha)
    except APIError as e:
        if e.status_code != 404: raise
        return None
    return base64.b64decode(r.content).decode()

async def gh_readme_paras(n=30, date='2022-06-01', min_words=50):
    "Rows from the READMEs of the `n` most-starred pre-cutoff repos, as of `date`"
    from ghapi.skill import GhApi
    from fastspec.errors import APIError
    api = GhApi()
    res = await api.search.repos(q=f'stars:>10000 created:<2021-01-01', sort='stars', per_page=n)
    rows = []
    for it in res['items']:
        try: txt = await gh_readme_asof(api, it.owner.login, it.name, date)
        except APIError as e:
            if e.status_code != 403: raise
            continue
        if txt: rows += doc_paras(txt, 'gh_readme', f'{it.full_name}@{date}', min_words)
    return rows

## Wikipedia, twice

Ordinary Wikipedia supplies competent encyclopedic prose. Simple English Wikipedia supplies the plainest register in any large corpus: short sentences, common words, written under rules close to the ones the meter enforces. Both fetchers take random article intros through the MediaWiki API. `wiki_pair` fetches the same title from both wikis, giving matched-content pairs that differ only in register.

In [ ]:
#| export
def _wiki_get(lang, **params):
    r = httpx.get(f'https://{lang}.wikipedia.org/w/api.php', follow_redirects=True,
        headers={'User-Agent': 'slopometer/0.1 (https://github.com/AnswerDotAI/slopometer)'},
        params=dict(action='query', prop='extracts', explaintext=1, exintro=1, format='json', **params))
    return r.json()['query']['pages'].values()

def wiki_paras(n=40, lang='en', min_words=50):
    "Rows from `n` random article intros in `lang` ('simple' for Simple English)"
    rows, seen = [], set()
    while len(seen) < n:
        for p in _wiki_get(lang, generator='random', grnnamespace=0, grnlimit=min(20, n-len(seen))):
            t = p.get('extract', '')
            if not t or p['title'] in seen: continue
            seen.add(p['title'])
            rows += doc_paras(t, f'wiki_{lang}', f'{lang}:{p["title"]}', min_words)
    return rows

def wiki_pair(title):
    "The intro of `title` from English and Simple English Wikipedia"
    return {lang: first(_wiki_get(lang, titles=title, redirects=1)).get('extract', '') for lang in ('en', 'simple')}

## The evaluation labels

The only meter-independent truth is the user's recorded dissatisfaction: a `;` prompt, or a complaint about clarity. `complaint_labels` finds those prompts and pairs each with the reply final it reacted to, which is the previous reply in the same session. Training never reads these rows. They exist to answer one question: does a scorer rank them above background?

In [ ]:
#| export
_complaint = re.compile(r"simple precise|plain english|too (long|verbose|wordy)|(can'?t|cannot|don'?t) follow|\brestate\b|\bshorter\b", re.I)

def complaint_labels(
    days=90, # How far back to walk the mirror
    path_re='-aai-ws', # Mirror filename filter
):
    "One row per reply final that drew a `;` or a clarity complaint"
    rows, prev, prevp = [], None, None
    for p, m in _mirror_prompts(days, path_re):
        if prevp != p: prev = None
        c = (m.content or '').strip()
        short = len(c.split()) <= 30 and not any(t in c for t in _notuser) and not c.startswith('#')
        if prev and (c == ';' or (len(c.split()) <= 3 and 'simplif' in c.lower()) or (short and _complaint.search(c))):
            rows.append(dict(text=prev, source='complaint', ref=f'{p}#{m.id}', prompt=c[:100]))
        txt = reply_final(m.out or '')
        prev, prevp = (txt if len(txt.split()) >= 10 else None), p
    return rows

## Assembly

`build_corpus` fetches every source, scores each row with `label_density`, applies the cuts, and writes two files under XDG state: `corpus.jsonl` for training and `eval.jsonl` for the complaint labels, which training must never read. The middle band joins neither side. Measured on this machine, reply densities run 31/64/124 at the 10th/50th/90th percentile, and the human sources run far lower but not at zero: your own long requests reach a median of 37, because genuine questions and dash habits fire rules there too. The default cuts come from those distributions: 50 keeps roughly half the replies as confident slop, and 15 keeps the cleanest quarter of the human paragraphs. Rows carry source, ref, and density, so an experiment can re-cut without refetching.

In [ ]:
#| export
from fastcore.parallel import parallel

def _densities(rows):
    "Attach `label_density` to each row, threaded"
    for r, d in zip(rows, parallel(label_density, [r['text'] for r in rows], n_workers=8, threadpool=True)): r['dens'] = d
    return rows

async def build_corpus(
    pos_cut=50, # Label density at or above which a reply counts as slop
    neg_cut=15, # Label density at or below which a human paragraph counts as clean
    wiki_n=150, # Random articles per wiki
    gh_n=60, # Most-starred repos to fetch
    out=None, # Output directory; default under XDG state
):
    "Fetch every source, apply the density cuts, and write `corpus.jsonl` and `eval.jsonl`"
    pos = _densities(mirror_replies())
    negs = _densities(readme_paras() + user_requests() + await gh_readme_paras(gh_n)
        + wiki_paras(wiki_n) + wiki_paras(wiki_n, lang='simple'))
    rows = [dict(**r, label='slop') for r in pos if r['dens'] >= pos_cut]
    rows += [dict(**r, label='clean') for r in negs if r['dens'] <= neg_cut]
    d = Path(out) if out else xdg_state_home()/'slopometer'
    d.mkdir(parents=True, exist_ok=True)
    (d/'corpus.jsonl').write_text('\n'.join(json.dumps(r) for r in rows))
    (d/'eval.jsonl').write_text('\n'.join(json.dumps(r) for r in complaint_labels()))
    return Counter((r['source'], r['label']) for r in rows)

In [ ]:
#| eval: false
counts = await build_corpus()
counts